# X1-REAL-1 Colab preparation and receipt gate

## Scope

This is a fail-closed, model-free Google Colab preparation notebook for the current `x_factor.x1_real_1` coordinator. It can inspect a supplied source release and candidate checkpoint, strictly parse supplied JSON, validate the frozen protocol and source closure, inventory checkpoint metadata, hash candidate files without loading them, and produce model-free gate, preflight, execution-plan, status, and packaging receipts.

It does not implement or run the scientific campaign. It does not deserialize a checkpoint, generate cohorts, fit a predictor, run outcomes or verifiers, perform model inference, train a model, commit custody phases, or perform analysis. Every status is blocked unless a separate released external runner supplies the required artifacts and execution authorization. The notebook always records `local_model_execution=False` and `training=False`.

The term **TFG is unresolved**. This notebook does not guess what TFG means and does not assume that it denotes a TPU, a GPU, a framework, or a model. It reports only directly detected Colab GPU, `torch_xla` TPU, or CPU state. GPU and TPU imports are optional runtime probes; neither Torch nor `torch_xla` is required for the model-free path.

## Exact setup

1. Open a fresh Colab notebook. Under **Runtime > Change runtime type**, any runtime can run the model-free checks. Select a GPU only if the operator intentionally wants the runtime receipt to report it. Do not select or interpret an accelerator on behalf of the unresolved TFG requirement. Do not install packages in this notebook.
2. Attach the supplied X1-REAL-1 source release as an unzipped directory under `/content`, or upload the directory before running this notebook. Automatic discovery accepts the release root at `/content` or one directory below `/content`. The selected root must contain both `pyproject.toml` and `x_factor/x1_real_1.py`. The notebook never clones, downloads, resets, overwrites, or deletes the source tree.
3. Optionally attach an already-produced artifact directory, candidate checkpoint, checkpoint registry, and release root. Stable artifact names are discovered recursively below `X1_ARTIFACT_ROOT`; multiple matches are a blocker. The checkpoint and all other supplied inputs remain outside the run root and are never packaged.
4. In a separate cell before this notebook's first code cell, set any needed overrides. Replace each example value with an actual path; do not set `X1_ARTIFACT_ROOT` or `X1_CHECKPOINT` unless those inputs are supplied.

```python
import os

os.environ['X1_SOURCE_ROOT'] = '/content/x1-real-1-source'
os.environ['X1_ARTIFACT_ROOT'] = '/content/x1-real-1-artifacts'
os.environ['X1_CHECKPOINT'] = '/content/checkpoints/candidate.bin'
os.environ['X1_REGISTRY'] = '/content/x1-real-1-artifacts/registry.json'
os.environ['X1_RUN_ROOT'] = '/content/x1-runs/operator-selected-run'
os.environ['X1_RUN_ID'] = 'operator-selected-run'
```

5. The supported overrides are `X1_SOURCE_ROOT`, `X1_ARTIFACT_ROOT`, `X1_CHECKPOINT`, `X1_RUN_ROOT`, `X1_RUN_ID`, and `X1_REGISTRY`. `X1_SOURCE_ROOT` takes precedence over attached-source discovery. If `X1_RUN_ROOT` is omitted, a unique run ID and run root are generated under `/content/x1_runs`. Reusing an explicit run root resumes immutable receipts and never overwrites an existing receipt.
6. Optional checkpoint-intake metadata variables are `X1_MODEL_CONFIG`, `X1_TOKENIZER_ARTIFACT`, `X1_RUNTIME_SOURCE_REVISION`, `X1_SOURCE_COMMIT`, `X1_GLOBAL_STEP`, `X1_STAGE`, `X1_PARAMETER_SHA256`, `X1_TOKENIZER_IDENTITY_SHA256`, `X1_READINESS_RECEIPT`, and `X1_READINESS_RECEIPT_SHA256`. Optional `X1_RELEASE_ROOT` defaults to the source root. The intake uses file hashes and these declared values only; file presence never promotes a candidate.
7. Run all code cells from top to bottom in a fresh kernel. Run **Runtime > Restart session** before reusing a notebook after changing the source. A CLI return code of `2` with an expected `BLOCKED` or invalid verdict is captured as evidence, not treated as a successful campaign. Review the final status and download the verified ZIP and SHA-256 receipt.

The notebook rejects duplicate JSON object keys, non-standard JSON constants, ambiguous stable artifact names, unsafe run identifiers, source/run overlap, and existing partial CLI state. It preserves each coordinator CLI invocation's stdout, stderr, and return code.

## Production runner prerequisites still missing

A separate, released external runner is required for actual X1-REAL-1 TPU/GPU execution. This repository does not provide an end-to-end model runner, cohort generator, predictor fitter, outcome producer, or verifier runner. Before any scientific execution, the operator must still supply and authorize: a release-pinned model-loading and generation implementation; a device-correct TPU/GPU adapter with pinned dependency provenance; deterministic independent cohort and task generators; development-only baseline and predictor fitting with artifact identities; structured outcome and verifier producers; custody and phase-commit operators; primary and checkpoint-replication analysis; storage, quota, timeout, and restart controls; and a documented external authorization/custody boundary. Until those exist, this notebook can only report preparation blockers and preserve receipts.

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import importlib.util
import json
import os
import platform
from pathlib import Path
import re
import shutil
import subprocess
import sys
from typing import Any

sys.dont_write_bytecode = True
if sys.version_info < (3, 10):
    raise RuntimeError('X1 Colab requires Python 3.10 or newer; runtime=' + platform.python_version())
CONTENT_ROOT = Path('/content').resolve()
CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
SAFE_TOKEN = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]{0,95}')
HASH64 = re.compile(r'[0-9a-f]{64}')
COMMIT40 = re.compile(r'[0-9a-f]{40}')

class DuplicateJSONKeyError(ValueError):
    pass

class ReceiptConflictError(RuntimeError):
    pass

def strict_object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise DuplicateJSONKeyError(f'duplicate JSON object key: {key!r}')
        result[key] = value
    return result

def reject_json_constant(value: str) -> None:
    raise ValueError(f'non-standard JSON constant is forbidden: {value}')

def strict_json_loads(payload: str, label: str) -> Any:
    try:
        return json.loads(
            payload,
            object_pairs_hook=strict_object,
            parse_constant=reject_json_constant,
        )
    except (json.JSONDecodeError, DuplicateJSONKeyError, ValueError) as exc:
        raise ValueError(f'{label}: {exc}') from exc

def strict_json_load(path: str | Path) -> Any:
    target = Path(path)
    return strict_json_loads(target.read_text(encoding='utf-8'), str(target))

def canonical_json_bytes(value: Any) -> bytes:
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False) + '\n').encode('utf-8')

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def write_bytes_no_overwrite(path: str | Path, payload: bytes) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'xb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())

def write_text_no_overwrite(path: str | Path, payload: str) -> None:
    write_bytes_no_overwrite(path, payload.encode('utf-8'))

def write_json_no_overwrite(path: str | Path, value: Any) -> None:
    write_bytes_no_overwrite(path, canonical_json_bytes(value))

def add_unique(items: list[str], value: str) -> None:
    if value not in items:
        items.append(value)

def configured_path(name: str) -> Path | None:
    raw = os.environ.get(name, '').strip()
    return Path(raw).expanduser().resolve() if raw else None

def paths_overlap(left: Path, right: Path) -> bool:
    return left == right or left.is_relative_to(right) or right.is_relative_to(left)

BOOTSTRAP_BLOCKERS: list[str] = []
requested_run_id = os.environ.get('X1_RUN_ID', '').strip()
if requested_run_id and not SAFE_TOKEN.fullmatch(requested_run_id):
    add_unique(BOOTSTRAP_BLOCKERS, 'X1_RUN_ID must match [A-Za-z0-9][A-Za-z0-9._-]{0,95}')
    requested_run_id = ''
run_root_override = configured_path('X1_RUN_ROOT')
if run_root_override is not None:
    RUN_ROOT = run_root_override
    if requested_run_id and RUN_ROOT.name != requested_run_id:
        add_unique(BOOTSTRAP_BLOCKERS, 'X1_RUN_ID must equal the final component of X1_RUN_ROOT')
    RUN_ID = requested_run_id or RUN_ROOT.name
else:
    RUN_ID = requested_run_id or f'x1-{__import__("datetime").datetime.now(__import__("datetime").timezone.utc).strftime("%Y%m%dT%H%M%SZ")}-{os.urandom(4).hex()}'
    RUN_ROOT = (CONTENT_ROOT / 'x1_runs' / RUN_ID).resolve()
    while not requested_run_id and RUN_ROOT.exists():
        RUN_ID = f'x1-{__import__("datetime").datetime.now(__import__("datetime").timezone.utc).strftime("%Y%m%dT%H%M%SZ")}-{os.urandom(4).hex()}'
        RUN_ROOT = (CONTENT_ROOT / 'x1_runs' / RUN_ID).resolve()
if RUN_ROOT == Path(RUN_ROOT.anchor):
    add_unique(BOOTSTRAP_BLOCKERS, 'X1_RUN_ROOT cannot be a filesystem root')
elif RUN_ROOT.exists() and not RUN_ROOT.is_dir():
    add_unique(BOOTSTRAP_BLOCKERS, f'run root is not a directory: {RUN_ROOT}')
else:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)

def persist_receipt(relative_name: str, value: Any) -> tuple[Any, bool]:
    target = (RUN_ROOT / relative_name).resolve()
    try:
        target.relative_to(RUN_ROOT)
    except ValueError as exc:
        raise ReceiptConflictError('receipt path escapes the run root') from exc
    if target.exists():
        return strict_json_load(target), True
    write_json_no_overwrite(target, value)
    return value, False

def source_requirements_met(path: Path) -> bool:
    return (
        path.is_dir()
        and not path.is_symlink()
        and (path / 'pyproject.toml').is_file()
        and not (path / 'pyproject.toml').is_symlink()
        and (path / 'x_factor' / 'x1_real_1.py').is_file()
        and not (path / 'x_factor' / 'x1_real_1.py').is_symlink()
    )

source_override = configured_path('X1_SOURCE_ROOT')
if source_override is not None:
    SOURCE_CANDIDATES = [source_override]
else:
    SOURCE_CANDIDATES = [CONTENT_ROOT]
    try:
        SOURCE_CANDIDATES.extend(sorted(path.resolve() for path in CONTENT_ROOT.iterdir() if path.is_dir() and not path.is_symlink()))
    except OSError as exc:
        add_unique(BOOTSTRAP_BLOCKERS, f'attached source discovery failed: {exc}')
valid_sources = [path for path in SOURCE_CANDIDATES if source_requirements_met(path)]
if source_override is not None and not valid_sources:
    add_unique(BOOTSTRAP_BLOCKERS, f'X1_SOURCE_ROOT lacks pyproject.toml and x_factor/x1_real_1.py: {source_override}')
elif source_override is None and not valid_sources:
    add_unique(BOOTSTRAP_BLOCKERS, 'no attached source tree with pyproject.toml and x_factor/x1_real_1.py was found at /content or one level below')
elif len(valid_sources) > 1:
    add_unique(BOOTSTRAP_BLOCKERS, 'multiple attached source trees match; set X1_SOURCE_ROOT: ' + ', '.join(str(path) for path in valid_sources))
SOURCE_ROOT = valid_sources[0] if len(valid_sources) == 1 else None

artifact_override = configured_path('X1_ARTIFACT_ROOT')
default_artifact_root = CONTENT_ROOT / 'x1_artifacts'
ARTIFACT_ROOT = artifact_override or (default_artifact_root.resolve() if default_artifact_root.is_dir() and not default_artifact_root.is_symlink() else None)
if ARTIFACT_ROOT is not None and (not ARTIFACT_ROOT.is_dir() or ARTIFACT_ROOT.is_symlink()):
    add_unique(BOOTSTRAP_BLOCKERS, f'X1_ARTIFACT_ROOT is not a regular directory: {ARTIFACT_ROOT}')
    ARTIFACT_ROOT = None

def env_file(name: str) -> Path | None:
    path = configured_path(name)
    if path is not None and (not path.is_file() or path.is_symlink()):
        add_unique(BOOTSTRAP_BLOCKERS, f'{name} is not a regular file: {path}')
        return None
    return path

CHECKPOINT_PATH = env_file('X1_CHECKPOINT')
MODEL_CONFIG_PATH = env_file('X1_MODEL_CONFIG')
TOKENIZER_ARTIFACT_PATH = env_file('X1_TOKENIZER_ARTIFACT')
READINESS_RECEIPT_PATH = env_file('X1_READINESS_RECEIPT')
REGISTRY_OVERRIDE_REQUESTED = bool(os.environ.get('X1_REGISTRY', '').strip())
REGISTRY_OVERRIDE = env_file('X1_REGISTRY')
RELEASE_ROOT_OVERRIDE = configured_path('X1_RELEASE_ROOT')
if RELEASE_ROOT_OVERRIDE is not None and (not RELEASE_ROOT_OVERRIDE.is_dir() or RELEASE_ROOT_OVERRIDE.is_symlink()):
    add_unique(BOOTSTRAP_BLOCKERS, f'X1_RELEASE_ROOT is not a regular directory: {RELEASE_ROOT_OVERRIDE}')
    RELEASE_ROOT_OVERRIDE = None
RELEASE_ROOT = RELEASE_ROOT_OVERRIDE or SOURCE_ROOT

RUNTIME_SOURCE_REVISION = os.environ.get('X1_RUNTIME_SOURCE_REVISION', '').strip() or None
SOURCE_COMMIT = os.environ.get('X1_SOURCE_COMMIT', '').strip() or None
STAGE = os.environ.get('X1_STAGE', '').strip() or None
PARAMETER_SHA256 = os.environ.get('X1_PARAMETER_SHA256', '').strip() or None
TOKENIZER_IDENTITY_SHA256 = os.environ.get('X1_TOKENIZER_IDENTITY_SHA256', '').strip() or None
READINESS_RECEIPT_SHA256 = os.environ.get('X1_READINESS_RECEIPT_SHA256', '').strip() or None
GLOBAL_STEP: int | None = None
global_step_raw = os.environ.get('X1_GLOBAL_STEP', '').strip()
if global_step_raw:
    if not global_step_raw.isdigit():
        add_unique(BOOTSTRAP_BLOCKERS, 'X1_GLOBAL_STEP must be a non-negative integer')
    else:
        GLOBAL_STEP = int(global_step_raw)
for name, value in (
    ('X1_PARAMETER_SHA256', PARAMETER_SHA256),
    ('X1_TOKENIZER_IDENTITY_SHA256', TOKENIZER_IDENTITY_SHA256),
    ('X1_READINESS_RECEIPT_SHA256', READINESS_RECEIPT_SHA256),
):
    if value is not None and not HASH64.fullmatch(value):
        add_unique(BOOTSTRAP_BLOCKERS, f'{name} must be a lowercase SHA-256 hex digest')
if SOURCE_COMMIT is not None and not COMMIT40.fullmatch(SOURCE_COMMIT):
    add_unique(BOOTSTRAP_BLOCKERS, 'X1_SOURCE_COMMIT must be a 40-character lowercase commit hash')

if SOURCE_ROOT is not None and paths_overlap(RUN_ROOT, SOURCE_ROOT):
    add_unique(BOOTSTRAP_BLOCKERS, 'run root and source root must not overlap')
    SOURCE_ROOT = None
if ARTIFACT_ROOT is not None and paths_overlap(RUN_ROOT, ARTIFACT_ROOT):
    add_unique(BOOTSTRAP_BLOCKERS, 'run root and artifact root must not overlap')
    ARTIFACT_ROOT = None
if CHECKPOINT_PATH is not None and RUN_ROOT.is_dir() and CHECKPOINT_PATH.is_relative_to(RUN_ROOT):
    add_unique(BOOTSTRAP_BLOCKERS, 'checkpoint must not be inside the run root because the run root is packaged')
    CHECKPOINT_PATH = None

setup_configuration = {
    'run_root': str(RUN_ROOT),
    'run_id': RUN_ID,
    'source_root': str(SOURCE_ROOT) if SOURCE_ROOT else None,
    'artifact_root': str(ARTIFACT_ROOT) if ARTIFACT_ROOT else None,
    'checkpoint': str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None,
    'registry_override': str(REGISTRY_OVERRIDE) if REGISTRY_OVERRIDE else None,
    'release_root': str(RELEASE_ROOT) if RELEASE_ROOT else None,
}
setup_report = {
    'schema': 'x1-real-1-colab-setup/v1',
    'status': 'BLOCKED' if BOOTSTRAP_BLOCKERS else 'READY_FOR_MODEL_FREE_INSPECTION',
    'scope': 'model_free_preparation_only',
    'local_model_execution': False,
    'training': False,
    'tfg_status': 'UNRESOLVED',
    'configuration': setup_configuration,
    'source_candidates': [str(path) for path in SOURCE_CANDIDATES],
    'blockers': list(BOOTSTRAP_BLOCKERS),
}
SETUP_RECEIPT, SETUP_RESUMED = persist_receipt('setup.json', setup_report)
if SETUP_RESUMED and SETUP_RECEIPT.get('configuration') != setup_configuration:
    add_unique(BOOTSTRAP_BLOCKERS, 'RESUME_CONFIGURATION_MISMATCH: existing setup.json describes different inputs')
print(json.dumps({'setup': SETUP_RECEIPT, 'resumed': SETUP_RESUMED, 'current_blockers': BOOTSTRAP_BLOCKERS}, indent=2, sort_keys=True))


In [ ]:
STABLE_ARTIFACT_NAMES = (
    'subject_manifest.json',
    'basis_qualification.json',
    'release_manifest.json',
    'primary_split.json',
    'basis_split.json',
    'development_split.json',
    'tasks.json',
    'predictor_identity.json',
    'baselines.json',
    'predictions.json',
    'prediction_receipt.json',
    'prediction_commit.json',
    'reveal_receipt.json',
    'reveal_commit.json',
    'outcomes.json',
    'evaluator_identity.json',
    'replication_receipt.json',
    'registry.json',
)
ACTIVE_BLOCKERS = list(BOOTSTRAP_BLOCKERS)
STRICT_RECORDS: list[dict[str, Any]] = []
STRICT_CACHE: dict[str, Any] = {}
STRICT_ERRORS: list[str] = []

def strict_document(path: str | Path) -> Any:
    target = Path(path)
    key = str(target.resolve())
    if key in STRICT_CACHE:
        return STRICT_CACHE[key]
    record: dict[str, Any] = {'path': key}
    try:
        if target.is_symlink() or not target.is_file():
            raise ValueError(f'not a regular JSON file: {target}')
        record['bytes'] = target.stat().st_size
        record['sha256'] = sha256_file(target)
        document = strict_json_load(target)
        record['status'] = 'VALID_STRICT_JSON'
        STRICT_CACHE[key] = document
    except (OSError, ValueError) as exc:
        record['status'] = 'REJECTED'
        record['error'] = f'{type(exc).__name__}: {exc}'
        STRICT_ERRORS.append(record['error'])
        STRICT_RECORDS.append(record)
        raise
    STRICT_RECORDS.append(record)
    return document

PROTOCOL_PATH = SOURCE_ROOT / 'x_factor' / 'protocols' / 'x1_real_1_v1.json' if SOURCE_ROOT else None
PROTOCOL_DOC: dict[str, Any] | None = None
if PROTOCOL_PATH is not None:
    try:
        loaded_protocol = strict_document(PROTOCOL_PATH)
        if not isinstance(loaded_protocol, dict):
            raise ValueError('frozen protocol JSON must be an object')
        PROTOCOL_DOC = loaded_protocol
    except (OSError, ValueError) as exc:
        add_unique(ACTIVE_BLOCKERS, f'PROTOCOL_JSON_REJECTED: {exc}')
else:
    add_unique(ACTIVE_BLOCKERS, 'PROTOCOL_UNAVAILABLE')

ALL_ARTIFACT_FILES: list[Path] = []
if ARTIFACT_ROOT is not None:
    for current, directories, filenames in os.walk(ARTIFACT_ROOT, followlinks=False):
        current_path = Path(current)
        directories[:] = sorted(name for name in directories if not (current_path / name).is_symlink())
        for filename in sorted(filenames):
            candidate = current_path / filename
            if candidate.is_symlink():
                if candidate.suffix.lower() == '.json' or filename in STABLE_ARTIFACT_NAMES:
                    add_unique(ACTIVE_BLOCKERS, f'SYMLINK_JSON_ARTIFACT_REJECTED: {candidate}')
                continue
            if candidate.is_file():
                ALL_ARTIFACT_FILES.append(candidate.resolve())
    for json_path in sorted({path for path in ALL_ARTIFACT_FILES if path.suffix.lower() == '.json'}):
        try:
            strict_document(json_path)
        except (OSError, ValueError):
            pass

ARTIFACT_MATCHES: dict[str, list[Path]] = {}
ARTIFACT_PATHS: dict[str, Path] = {}
ARTIFACT_DOCS: dict[str, Any] = {}
ARTIFACT_INVENTORY: dict[str, Any] = {}
for name in STABLE_ARTIFACT_NAMES:
    matches = sorted(path for path in ALL_ARTIFACT_FILES if path.name == name)
    ARTIFACT_MATCHES[name] = matches
    record: dict[str, Any] = {'status': 'MISSING', 'matches': [str(path) for path in matches]}
    if len(matches) > 1:
        add_unique(ACTIVE_BLOCKERS, f'AMBIGUOUS_STABLE_ARTIFACT:{name}')
        record['status'] = 'AMBIGUOUS'
    elif len(matches) == 1:
        path = matches[0]
        record.update({'path': str(path), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
        try:
            document = strict_document(path)
        except (OSError, ValueError) as exc:
            record.update({'status': 'REJECTED', 'error': f'{type(exc).__name__}: {exc}'})
        else:
            ARTIFACT_PATHS[name] = path
            ARTIFACT_DOCS[name] = document
            record['status'] = 'STRICTLY_LOADED'
    ARTIFACT_INVENTORY[name] = record

registry_artifact_present = 'registry.json' in ARTIFACT_MATCHES
if REGISTRY_OVERRIDE_REQUESTED:
    REGISTRY_PATH = REGISTRY_OVERRIDE
elif registry_artifact_present:
    REGISTRY_PATH = ARTIFACT_PATHS.get('registry.json')
else:
    REGISTRY_PATH = SOURCE_ROOT / 'x_factor' / 'registry' / 'checkpoints.json' if SOURCE_ROOT else None
REGISTRY_DOC: Any = None
if REGISTRY_PATH is not None:
    try:
        REGISTRY_DOC = strict_document(REGISTRY_PATH)
    except (OSError, ValueError) as exc:
        add_unique(ACTIVE_BLOCKERS, f'REGISTRY_JSON_REJECTED: {exc}')
else:
    add_unique(ACTIVE_BLOCKERS, 'CHECKPOINT_REGISTRY_UNAVAILABLE')

def referenced_json_paths(value: Any, key: str = '') -> list[Path]:
    found: list[Path] = []
    if isinstance(value, dict):
        for child_key, child in value.items():
            found.extend(referenced_json_paths(child, str(child_key)))
    elif isinstance(value, list):
        for child in value:
            found.extend(referenced_json_paths(child, key))
    elif isinstance(value, str) and 'path' in key.lower() and value.lower().endswith('.json'):
        candidate = Path(value).expanduser()
        roots = [root for root in (ARTIFACT_ROOT, RELEASE_ROOT, SOURCE_ROOT) if root is not None]
        if not candidate.is_absolute():
            candidate = next((root / candidate for root in roots if (root / candidate).is_file()), candidate)
        if candidate.is_file() or candidate.is_symlink():
            found.append(candidate)
    return found

reference_queue: list[Any] = [PROTOCOL_DOC, REGISTRY_DOC, *ARTIFACT_DOCS.values()]
reference_seen: set[Path] = set()
while reference_queue:
    document = reference_queue.pop()
    for referenced in referenced_json_paths(document):
        resolved = referenced.resolve()
        if resolved in reference_seen:
            continue
        reference_seen.add(resolved)
        if referenced.is_symlink():
            add_unique(ACTIVE_BLOCKERS, f'SYMLINK_JSON_REFERENCE_REJECTED: {referenced}')
            continue
        try:
            reference_queue.append(strict_document(referenced))
        except (OSError, ValueError):
            pass

for error in STRICT_ERRORS:
    add_unique(ACTIVE_BLOCKERS, f'STRICT_JSON_REJECTED: {error}')
STRICT_AUDIT_REPORT = {
    'schema': 'x1-real-1-colab-strict-json-audit/v1',
    'duplicate_keys_rejected': True,
    'nonstandard_constants_rejected': True,
    'protocol_path': str(PROTOCOL_PATH) if PROTOCOL_PATH else None,
    'registry_path': str(REGISTRY_PATH) if REGISTRY_PATH else None,
    'files': STRICT_RECORDS,
    'errors': STRICT_ERRORS,
}
persist_receipt('strict_json_audit.json', STRICT_AUDIT_REPORT)
persist_receipt('artifact_inventory.json', {
    'schema': 'x1-real-1-colab-artifact-inventory/v1',
    'artifact_root': str(ARTIFACT_ROOT) if ARTIFACT_ROOT else None,
    'stable_names': list(STABLE_ARTIFACT_NAMES),
    'artifacts': ARTIFACT_INVENTORY,
    'all_regular_file_count': len(ALL_ARTIFACT_FILES),
    'all_regular_file_bytes': sum(path.stat().st_size for path in ALL_ARTIFACT_FILES),
    'checkpoint_packaged': False,
})
print(json.dumps({'strict_json': STRICT_AUDIT_REPORT, 'artifacts': ARTIFACT_INVENTORY, 'blockers': ACTIVE_BLOCKERS}, indent=2, sort_keys=True))


In [ ]:
COORDINATOR = None
COORDINATOR_IMPORT_ERROR: str | None = None
if SOURCE_ROOT is not None:
    loaded_x_factor = {
        name: module for name, module in sys.modules.items()
        if name == 'x_factor' or name.startswith('x_factor.')
    }
    outside_source = []
    for name, module in loaded_x_factor.items():
        module_file = getattr(module, '__file__', None)
        if module_file and not Path(module_file).resolve().is_relative_to(SOURCE_ROOT):
            outside_source.append(name)
    if outside_source:
        COORDINATOR_IMPORT_ERROR = 'x_factor was imported before source selection from outside the supplied source: ' + ', '.join(sorted(outside_source))
    else:
        if str(SOURCE_ROOT) not in sys.path:
            sys.path.insert(0, str(SOURCE_ROOT))
        try:
            COORDINATOR = importlib.import_module('x_factor.x1_real_1')
            module_path = Path(COORDINATOR.__file__).resolve()
            expected_path = (SOURCE_ROOT / 'x_factor' / 'x1_real_1.py').resolve()
            if module_path != expected_path:
                COORDINATOR = None
                COORDINATOR_IMPORT_ERROR = f'coordinator module resolved to unexpected file: {module_path}'
        except Exception as exc:
            COORDINATOR_IMPORT_ERROR = f'{type(exc).__name__}: {exc}'
if COORDINATOR_IMPORT_ERROR:
    add_unique(ACTIVE_BLOCKERS, f'COORDINATOR_IMPORT_BLOCKED: {COORDINATOR_IMPORT_ERROR}')

def optional_spec(name: str) -> Any:
    try:
        return importlib.util.find_spec(name)
    except (ImportError, ValueError):
        return None

colab_available = optional_spec('google.colab') is not None
gpu_devices: list[dict[str, Any]] = []
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    try:
        gpu_probe = subprocess.run(
            [nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'],
            capture_output=True,
            check=False,
            timeout=20,
        )
        if gpu_probe.returncode == 0:
            for line in gpu_probe.stdout.decode('utf-8', errors='replace').splitlines():
                fields = [field.strip() for field in line.split(',')]
                if fields and fields[0]:
                    record = {'name': fields[0]}
                    if len(fields) > 1:
                        record['memory_total_mib'] = fields[1]
                    if len(fields) > 2:
                        record['driver_version'] = fields[2]
                    gpu_devices.append(record)
    except (OSError, subprocess.SubprocessError):
        pass

torch_report: dict[str, Any] = {'available': False}
if optional_spec('torch') is not None:
    try:
        torch_module = importlib.import_module('torch')
        torch_report.update({
            'available': True,
            'version': getattr(torch_module, '__version__', None),
            'cuda_available': bool(torch_module.cuda.is_available()),
            'cuda_device_count': int(torch_module.cuda.device_count()) if torch_module.cuda.is_available() else 0,
        })
    except Exception as exc:
        torch_report.update({'import_error': f'{type(exc).__name__}: {exc}'})

xla_report: dict[str, Any] = {'available': False}
if optional_spec('torch_xla') is not None:
    try:
        xla_module = importlib.import_module('torch_xla')
        xla_report['available'] = True
        xla_report['version'] = getattr(xla_module, '__version__', None)
        hardware_probe = getattr(xla_module, 'xla_device_hw', None)
        if callable(hardware_probe):
            xla_report['device_type'] = str(hardware_probe())
    except Exception as exc:
        xla_report.update({'available': False, 'import_error': f'{type(exc).__name__}: {exc}'})

xla_type = str(xla_report.get('device_type', '')).upper()
if gpu_devices or torch_report.get('cuda_available'):
    accelerator = {'kind': 'COLAB_GPU' if colab_available else 'GPU', 'devices': gpu_devices, 'verified_by': 'nvidia-smi/torch metadata'}
elif 'TPU' in xla_type:
    accelerator = {'kind': 'TORCH_XLA_TPU', 'devices': gpu_devices, 'verified_by': 'torch_xla.xla_device_hw'}
else:
    accelerator = {'kind': 'CPU', 'devices': [], 'verified_by': 'no GPU or torch_xla TPU probe reported an accelerator'}
disk = shutil.disk_usage(CONTENT_ROOT)
runtime_report = {
    'schema': 'x1-real-1-colab-runtime/v1',
    'colab_runtime': colab_available,
    'python_version': sys.version.split()[0],
    'platform': sys.platform,
    'accelerator': accelerator,
    'torch': torch_report,
    'torch_xla': xla_report,
    'torch_required_for_model_free_path': False,
    'tfg': {'status': 'UNRESOLVED', 'meaning_assumed': False},
    'coordinator_module': str(Path(COORDINATOR.__file__).resolve()) if COORDINATOR else None,
    'coordinator_import_error': COORDINATOR_IMPORT_ERROR,
    'storage': {
        'content_path': str(CONTENT_ROOT),
        'total_bytes': disk.total,
        'used_bytes': disk.used,
        'free_bytes': disk.free,
    },
    'local_model_execution': False,
    'training': False,
}
RUNTIME_REPORT, RUNTIME_RESUMED = persist_receipt('runtime.json', runtime_report)
print(json.dumps({'runtime': RUNTIME_REPORT, 'resumed': RUNTIME_RESUMED, 'blockers': ACTIVE_BLOCKERS}, indent=2, sort_keys=True))


In [ ]:
def cli_timeout_seconds() -> int:
    raw = os.environ.get('X1_CLI_TIMEOUT_SECONDS', '900').strip()
    try:
        value = int(raw)
    except ValueError:
        value = 900
    return value if 30 <= value <= 86400 else 900

def run_coordinator_cli(
    name: str,
    arguments: list[str],
    *,
    accepted_returncodes: tuple[int, ...] = (0,),
    expected_statuses: tuple[str, ...] = (),
) -> dict[str, Any]:
    if not SAFE_TOKEN.fullmatch(name):
        raise ValueError('unsafe CLI receipt name')
    if SOURCE_ROOT is None or COORDINATOR is None:
        raise RuntimeError('coordinator is unavailable')
    receipt_path = RUN_ROOT / f'{name}.json'
    cli_root = RUN_ROOT / 'cli'
    cli_root.mkdir(parents=True, exist_ok=True)
    stdout_path = cli_root / f'{name}.stdout.txt'
    stderr_path = cli_root / f'{name}.stderr.txt'
    execution_path = cli_root / f'{name}.execution.json'
    invocation_path = cli_root / f'{name}.invocation.json'
    capture_paths = (stdout_path, stderr_path, execution_path)
    command = [sys.executable, '-m', 'x_factor.x1_real_1', *[str(item) for item in arguments]]

    if receipt_path.exists() or any(path.exists() for path in capture_paths):
        receipt = strict_json_load(receipt_path) if receipt_path.exists() else None
        execution = strict_json_load(execution_path) if execution_path.exists() else None
        stdout_text = stdout_path.read_text(encoding='utf-8') if stdout_path.exists() else None
        stderr_text = stderr_path.read_text(encoding='utf-8', errors='replace') if stderr_path.exists() else None
        stdout_document = strict_json_loads(stdout_text, str(stdout_path)) if stdout_text is not None else None
        command_match = bool(execution and execution.get('command') == command)
        stdout_matches = bool(receipt is not None and stdout_document == receipt)
        returncode = execution.get('returncode') if execution else None
        status = receipt.get('status') if isinstance(receipt, dict) else None
        accepted = returncode in accepted_returncodes and receipt is not None and command_match and stdout_matches
        invocation = {
            'schema': 'x1-real-1-colab-cli-invocation/v1',
            'name': name,
            'command': command,
            'cwd': str(SOURCE_ROOT),
            'returncode': returncode,
            'status': status,
            'accepted_return_code': accepted,
            'expected_blocked_result': returncode == 2 and status in expected_statuses,
            'command_match': command_match,
            'stdout_matches_receipt': stdout_matches,
            'receipt_path': str(receipt_path),
            'stdout_path': str(stdout_path),
            'stderr_path': str(stderr_path),
            'execution_path': str(execution_path),
            'resumed': True,
            'partial_state': receipt is None,
            'error': execution.get('error') if execution else 'existing state lacks complete CLI captures',
        }
        if not invocation_path.exists():
            write_json_no_overwrite(invocation_path, invocation)
        return {**invocation, 'receipt': receipt}

    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(SOURCE_ROOT) + (os.pathsep + environment['PYTHONPATH'] if environment.get('PYTHONPATH') else '')
    environment['PYTHONDONTWRITEBYTECODE'] = '1'
    process_error: str | None = None
    try:
        process = subprocess.run(
            command,
            cwd=str(SOURCE_ROOT),
            env=environment,
            capture_output=True,
            check=False,
            timeout=cli_timeout_seconds(),
        )
        returncode = process.returncode
        stdout_bytes = process.stdout
        stderr_bytes = process.stderr
    except subprocess.TimeoutExpired as exc:
        returncode = None
        stdout_bytes = exc.stdout or b''
        stderr_bytes = exc.stderr or b''
        process_error = 'subprocess timeout'
    except OSError as exc:
        returncode = None
        stdout_bytes = b''
        stderr_bytes = b''
        process_error = f'{type(exc).__name__}: {exc}'
    write_bytes_no_overwrite(stdout_path, stdout_bytes)
    write_bytes_no_overwrite(stderr_path, stderr_bytes)
    execution = {
    'schema': 'x1-real-1-colab-cli-execution/v1',
    'command': command,
    'cwd': str(SOURCE_ROOT),
    'returncode': returncode,
    'error': process_error,
}
    write_json_no_overwrite(execution_path, execution)
    parse_error: str | None = None
    stdout_document: Any = None
    try:
        stdout_document = strict_json_loads(stdout_bytes.decode('utf-8'), str(stdout_path))
    except (UnicodeDecodeError, ValueError) as exc:
        parse_error = f'{type(exc).__name__}: {exc}'
    receipt = strict_json_load(receipt_path) if receipt_path.exists() else None
    if receipt is None and stdout_document is not None and not receipt_path.exists():
        write_json_no_overwrite(receipt_path, stdout_document)
        receipt = stdout_document
    stdout_matches = receipt is not None and stdout_document == receipt
    status = receipt.get('status') if isinstance(receipt, dict) else None
    accepted = returncode in accepted_returncodes and receipt is not None and stdout_matches
    invocation = {
    'schema': 'x1-real-1-colab-cli-invocation/v1',
    'name': name,
    'command': command,
    'cwd': str(SOURCE_ROOT),
    'returncode': returncode,
    'status': status,
    'accepted_return_code': accepted,
    'expected_blocked_result': returncode == 2 and status in expected_statuses,
    'command_match': True,
    'stdout_matches_receipt': stdout_matches,
    'receipt_path': str(receipt_path),
    'stdout_path': str(stdout_path),
    'stderr_path': str(stderr_path),
    'execution_path': str(execution_path),
    'resumed': False,
    'partial_state': receipt is None,
    'error': process_error or parse_error,
}
    write_json_no_overwrite(invocation_path, invocation)
    return {**invocation, 'receipt': receipt}

def operation_unavailable(name: str, blockers: list[str]) -> dict[str, Any]:
    report = {
    'schema': 'x1-real-1-colab-operation-unavailable/v1',
    'status': 'BLOCKED',
    'operation': name,
    'blockers': list(dict.fromkeys(blockers)),
    'local_model_execution': False,
    'training': False,
    }
    persisted, _ = persist_receipt(f'{name}_unavailable.json', report)
    return {'receipt': persisted, 'accepted_return_code': False, 'unavailable': True, 'returncode': None, 'status': 'BLOCKED'}


In [ ]:
if COORDINATOR is not None and PROTOCOL_PATH is not None and PROTOCOL_DOC is not None:
    PROTOCOL_INVOCATION = run_coordinator_cli(
        'protocol_validation',
        [
            'validate-protocol',
            '--protocol', str(PROTOCOL_PATH),
            '--repo-root', str(SOURCE_ROOT),
            '--check-source-closure',
        ],
        accepted_returncodes=(0, 2),
    )
else:
    PROTOCOL_INVOCATION = operation_unavailable('protocol_validation', ACTIVE_BLOCKERS)
PROTOCOL_REPORT = PROTOCOL_INVOCATION.get('receipt')
print(json.dumps({
    'returncode': PROTOCOL_INVOCATION.get('returncode'),
    'accepted_report': PROTOCOL_INVOCATION.get('accepted_return_code'),
    'valid': PROTOCOL_REPORT.get('valid') if isinstance(PROTOCOL_REPORT, dict) else None,
    'protocol_sha256': PROTOCOL_REPORT.get('protocol_sha256') if isinstance(PROTOCOL_REPORT, dict) else None,
    'errors': PROTOCOL_REPORT.get('errors', []) if isinstance(PROTOCOL_REPORT, dict) else ACTIVE_BLOCKERS,
}, indent=2, sort_keys=True))


In [ ]:
if COORDINATOR is not None and REGISTRY_PATH is not None and REGISTRY_DOC is not None:
    INVENTORY_INVOCATION = run_coordinator_cli(
        'checkpoint_inventory',
        ['inventory', '--registry', str(REGISTRY_PATH), '--verify-files'],
        accepted_returncodes=(0, 2),
        expected_statuses=('ELIGIBLE_SUBJECT_AVAILABLE', 'NO_ELIGIBLE_CHECKPOINT'),
    )
else:
    INVENTORY_INVOCATION = operation_unavailable('checkpoint_inventory', ACTIVE_BLOCKERS + ['REGISTRY_UNAVAILABLE'])
INVENTORY_REPORT = INVENTORY_INVOCATION.get('receipt')

if COORDINATOR is not None and CHECKPOINT_PATH is not None:
    intake_arguments = ['intake', '--checkpoint', str(CHECKPOINT_PATH)]
    if MODEL_CONFIG_PATH is not None:
        intake_arguments.extend(['--config', str(MODEL_CONFIG_PATH)])
    if TOKENIZER_ARTIFACT_PATH is not None:
        intake_arguments.extend(['--tokenizer', str(TOKENIZER_ARTIFACT_PATH)])
    if RUNTIME_SOURCE_REVISION is not None:
        intake_arguments.extend(['--runtime-source-revision', RUNTIME_SOURCE_REVISION])
    if SOURCE_COMMIT is not None:
        intake_arguments.extend(['--source-commit', SOURCE_COMMIT])
    if GLOBAL_STEP is not None:
        intake_arguments.extend(['--global-step', str(GLOBAL_STEP)])
    if STAGE is not None:
        intake_arguments.extend(['--stage', STAGE])
    if PARAMETER_SHA256 is not None:
        intake_arguments.extend(['--parameter-sha256', PARAMETER_SHA256])
    if TOKENIZER_IDENTITY_SHA256 is not None:
        intake_arguments.extend(['--tokenizer-identity-sha256', TOKENIZER_IDENTITY_SHA256])
    if READINESS_RECEIPT_PATH is not None:
        intake_arguments.extend(['--readiness-receipt', str(READINESS_RECEIPT_PATH)])
    if READINESS_RECEIPT_SHA256 is not None:
        intake_arguments.extend(['--readiness-receipt-sha256', READINESS_RECEIPT_SHA256])
    intake_arguments.extend(['--out', str(RUN_ROOT / 'candidate_intake.json')])
    INTAKE_INVOCATION = run_coordinator_cli(
        'candidate_intake',
        intake_arguments,
        accepted_returncodes=(0, 2),
        expected_statuses=('INVENTORIED', 'UNQUALIFIED_NEW'),
    )
else:
    INTAKE_INVOCATION = operation_unavailable('candidate_intake', ACTIVE_BLOCKERS + ['CHECKPOINT_NOT_CONFIGURED_OR_INVALID'])
INTAKE_REPORT = INTAKE_INVOCATION.get('receipt')
print(json.dumps({
    'inventory': {
        'returncode': INVENTORY_INVOCATION.get('returncode'),
        'status': INVENTORY_REPORT.get('status') if isinstance(INVENTORY_REPORT, dict) else None,
        'eligible_candidates': INVENTORY_REPORT.get('eligible_candidates') if isinstance(INVENTORY_REPORT, dict) else None,
    },
    'candidate_intake': {
        'returncode': INTAKE_INVOCATION.get('returncode'),
        'status': INTAKE_REPORT.get('status') if isinstance(INTAKE_REPORT, dict) else None,
        'checkpoint_file_sha256': INTAKE_REPORT.get('checkpoint_file_sha256') if isinstance(INTAKE_REPORT, dict) else None,
        'missing_fields': INTAKE_REPORT.get('missing_fields') if isinstance(INTAKE_REPORT, dict) else None,
        'research_subject': INTAKE_REPORT.get('research_subject') if isinstance(INTAKE_REPORT, dict) else None,
        'eligible_for_x1': INTAKE_REPORT.get('eligible_for_x1') if isinstance(INTAKE_REPORT, dict) else None,
        'promotion_status': INTAKE_REPORT.get('promotion_status') if isinstance(INTAKE_REPORT, dict) else None,
        'model_execution': INTAKE_REPORT.get('model_execution') if isinstance(INTAKE_REPORT, dict) else None,
    },
    'notice': 'Intake hashes files and records declared metadata only. It never deserializes, promotes, infers, or trains on the candidate.',
}, indent=2, sort_keys=True))


In [ ]:
if COORDINATOR is not None and PROTOCOL_PATH is not None and PROTOCOL_DOC is not None and REGISTRY_PATH is not None and REGISTRY_DOC is not None:
    gate_arguments = ['gate', '--registry', str(REGISTRY_PATH), '--protocol', str(PROTOCOL_PATH)]
    if 'basis_qualification.json' in ARTIFACT_PATHS:
        gate_arguments.extend(['--basis-qualification', str(ARTIFACT_PATHS['basis_qualification.json'])])
    if 'release_manifest.json' in ARTIFACT_PATHS:
        gate_arguments.extend(['--release-manifest', str(ARTIFACT_PATHS['release_manifest.json'])])
    if RELEASE_ROOT is not None:
        gate_arguments.extend(['--release-root', str(RELEASE_ROOT)])
    GATE_INVOCATION = run_coordinator_cli(
        'prerequisite_gate',
        gate_arguments,
        accepted_returncodes=(0, 2),
        expected_statuses=('PROTOCOL_INVALID', 'NO_ELIGIBLE_CHECKPOINT', 'BASIS_NOT_QUALIFIED', 'RELEASE_MANIFEST_REQUIRED', 'READY_FOR_EXTERNAL_PHASE_1'),
    )
else:
    GATE_INVOCATION = operation_unavailable('prerequisite_gate', ACTIVE_BLOCKERS)
GATE_REPORT = GATE_INVOCATION.get('receipt')
print(json.dumps({
    'returncode': GATE_INVOCATION.get('returncode'),
    'accepted_report': GATE_INVOCATION.get('accepted_return_code'),
    'status': GATE_REPORT.get('status') if isinstance(GATE_REPORT, dict) else None,
    'blockers': GATE_REPORT.get('blockers') if isinstance(GATE_REPORT, dict) else ACTIVE_BLOCKERS,
    'next_operator_action': GATE_REPORT.get('next_operator_action') if isinstance(GATE_REPORT, dict) else None,
}, indent=2, sort_keys=True))


In [ ]:
if COORDINATOR is not None and PROTOCOL_PATH is not None and PROTOCOL_DOC is not None:
    preflight_arguments = [
        'preflight',
        '--protocol', str(PROTOCOL_PATH),
        '--source-root', str(SOURCE_ROOT),
        '--out', str(RUN_ROOT / 'preflight.json'),
    ]
    artifact_options = {
        'subject_manifest.json': '--subject',
        'basis_qualification.json': '--basis-qualification',
        'release_manifest.json': '--release-manifest',
        'primary_split.json': '--split',
        'basis_split.json': '--basis-split',
        'development_split.json': '--development-split',
        'tasks.json': '--tasks',
        'predictor_identity.json': '--predictor-identity',
        'baselines.json': '--baselines',
        'predictions.json': '--predictions',
        'prediction_receipt.json': '--prediction',
        'prediction_commit.json': '--prediction-commit',
        'reveal_receipt.json': '--reveal',
        'reveal_commit.json': '--reveal-commit',
        'outcomes.json': '--outcomes',
        'evaluator_identity.json': '--evaluator-identity',
        'replication_receipt.json': '--replication',
    }
    for name, option in artifact_options.items():
        if name in ARTIFACT_PATHS:
            preflight_arguments.extend([option, str(ARTIFACT_PATHS[name])])
    if REGISTRY_PATH is not None and REGISTRY_DOC is not None:
        preflight_arguments.extend(['--registry', str(REGISTRY_PATH)])
    checkpoint_root = CHECKPOINT_PATH.parent if CHECKPOINT_PATH is not None else (ARTIFACT_ROOT or RELEASE_ROOT or SOURCE_ROOT)
    if checkpoint_root is not None:
        preflight_arguments.extend(['--checkpoint-root', str(checkpoint_root)])
    if RELEASE_ROOT is not None:
        preflight_arguments.extend(['--release-root', str(RELEASE_ROOT)])
    PREFLIGHT_INVOCATION = run_coordinator_cli(
        'preflight',
        preflight_arguments,
        accepted_returncodes=(0, 2),
        expected_statuses=('BLOCKED', 'READY_FOR_EXTERNAL_PREDICTION'),
    )
else:
    PREFLIGHT_INVOCATION = operation_unavailable('preflight', ACTIVE_BLOCKERS)
PREFLIGHT_REPORT = PREFLIGHT_INVOCATION.get('receipt')
failed_checks = []
if isinstance(PREFLIGHT_REPORT, dict):
    failed_checks = [
        {'id': check.get('id'), 'status': check.get('status'), 'errors': check.get('errors', [])}
        for check in PREFLIGHT_REPORT.get('checks', [])
        if check.get('status') in {'FAIL', 'BLOCKED'}
    ]
print(json.dumps({
    'returncode': PREFLIGHT_INVOCATION.get('returncode'),
    'accepted_report': PREFLIGHT_INVOCATION.get('accepted_return_code'),
    'expected_blocked_result': PREFLIGHT_INVOCATION.get('expected_blocked_result'),
    'status': PREFLIGHT_REPORT.get('status') if isinstance(PREFLIGHT_REPORT, dict) else None,
    'check_summary': PREFLIGHT_REPORT.get('check_summary') if isinstance(PREFLIGHT_REPORT, dict) else None,
    'blockers': PREFLIGHT_REPORT.get('blockers') if isinstance(PREFLIGHT_REPORT, dict) else ACTIVE_BLOCKERS,
    'failed_checks': failed_checks,
    'external_compute_authorized': PREFLIGHT_REPORT.get('external_compute_authorized') if isinstance(PREFLIGHT_REPORT, dict) else False,
    'local_model_execution': False,
    'training': False,
}, indent=2, sort_keys=True))


In [ ]:
preflight_path = RUN_ROOT / 'preflight.json'
intake_path = RUN_ROOT / 'candidate_intake.json'
preflight_strictly_loaded = False
intake_strictly_loaded = False
if preflight_path.exists():
    try:
        strict_json_load(preflight_path)
        preflight_strictly_loaded = True
    except (OSError, ValueError) as exc:
        add_unique(ACTIVE_BLOCKERS, f'PREFLIGHT_RECEIPT_JSON_REJECTED: {exc}')
if intake_path.exists():
    try:
        strict_json_load(intake_path)
        intake_strictly_loaded = True
    except (OSError, ValueError) as exc:
        add_unique(ACTIVE_BLOCKERS, f'INTAKE_RECEIPT_JSON_REJECTED: {exc}')
if COORDINATOR is not None and PROTOCOL_PATH is not None and PROTOCOL_DOC is not None and preflight_strictly_loaded:
    plan_arguments = [
        'execution-plan',
        '--protocol', str(PROTOCOL_PATH),
        '--preflight', str(preflight_path),
        '--source-root', str(SOURCE_ROOT),
        '--out', str(RUN_ROOT / 'execution_plan.json'),
    ]
    if intake_strictly_loaded:
        plan_arguments.extend(['--intake', str(intake_path)])
    EXECUTION_PLAN_INVOCATION = run_coordinator_cli(
        'execution_plan',
        plan_arguments,
        accepted_returncodes=(0, 2),
        expected_statuses=('BLOCKED', 'BLOCKED_POWER_DEVIATION_REQUIRED', 'READY_FOR_EXTERNAL_EXECUTION'),
    )
else:
    EXECUTION_PLAN_INVOCATION = operation_unavailable('execution_plan', ACTIVE_BLOCKERS + ['STRICT_PREFLIGHT_RECEIPT_UNAVAILABLE'])
EXECUTION_PLAN_REPORT = EXECUTION_PLAN_INVOCATION.get('receipt')
print(json.dumps({
    'returncode': EXECUTION_PLAN_INVOCATION.get('returncode'),
    'accepted_report': EXECUTION_PLAN_INVOCATION.get('accepted_return_code'),
    'expected_blocked_result': EXECUTION_PLAN_INVOCATION.get('expected_blocked_result'),
    'status': EXECUTION_PLAN_REPORT.get('status') if isinstance(EXECUTION_PLAN_REPORT, dict) else None,
    'blockers': EXECUTION_PLAN_REPORT.get('blockers') if isinstance(EXECUTION_PLAN_REPORT, dict) else ACTIVE_BLOCKERS,
    'external_execution_required': EXECUTION_PLAN_REPORT.get('external_execution_required') if isinstance(EXECUTION_PLAN_REPORT, dict) else True,
    'external_compute_authorized': EXECUTION_PLAN_REPORT.get('external_compute_authorized') if isinstance(EXECUTION_PLAN_REPORT, dict) else False,
    'local_model_execution': False,
    'training': False,
}, indent=2, sort_keys=True))


## Optional receipt-only custody actions

The next cell defines, but does not invoke, guarded helpers for `commit-prediction`, `commit-reveal`, `commit-phase`, and `analyze`. Each action defaults to `False`. An operator must explicitly enable that action, set `X1_OPERATOR_AUTHORIZATION` to the exact phrase shown in the cell, and supply an argument list whose required options point to already-existing files. Outputs must be new paths under the immutable run root. The helper allowlist excludes inference, training, cohort generation, predictor fitting, outcome generation, and verifier execution; it only coordinates already-produced receipt artifacts.

In [ ]:
OPTIONAL_AUTHORIZATION_PHRASE = 'AUTHORIZE_RECEIPT_ONLY_COORDINATION'
OPTIONAL_ACTIONS_ENABLED = {
    'commit-prediction': False,
    'commit-reveal': False,
    'commit-phase': False,
    'analyze': False,
}
OPTIONAL_ACTION_INVOCATIONS: dict[str, Any] = {}
OPTIONAL_FILE_OPTIONS = {
    '--receipt', '--artifact-path', '--previous-commit', '--protocol', '--subject',
    '--split', '--tasks', '--predictions', '--baselines', '--predictor-identity',
    '--basis-qualification', '--release-manifest', '--artifact-root', '--prediction',
    '--outcomes', '--evaluator-identity', '--prediction-commit', '--reveal-commit',
    '--replication', '--primary-bundle', '--replication-bundle', '--development-split',
    '--basis-split', '--preflight', '--intake', '--registry',
}
OPTIONAL_DIRECTORY_OPTIONS = {'--source-root', '--release-root', '--checkpoint-root'}
OPTIONAL_REQUIRED_OPTIONS = {
    'commit-prediction': {
        '--protocol', '--subject', '--split', '--tasks', '--predictions', '--baselines',
        '--predictor-identity', '--basis-qualification', '--cohort-id', '--development-split',
        '--basis-split', '--release-manifest', '--out',
    },
    'commit-reveal': {
        '--protocol', '--subject', '--split', '--tasks', '--basis-qualification',
        '--release-manifest', '--prediction', '--prediction-commit', '--outcomes',
        '--evaluator-identity', '--out',
    },
    'commit-phase': {
        '--receipt', '--artifact-path', '--phase', '--custodian', '--attestation-sha256', '--out',
    },
    'analyze': {
        '--protocol', '--subject', '--split', '--tasks', '--prediction', '--prediction-commit',
        '--reveal', '--reveal-commit', '--basis-qualification', '--release-manifest',
        '--basis-split', '--development-split', '--out',
    },
}

def run_optional_receipt_action(action: str, arguments: list[str]) -> dict[str, Any]:
    if action not in OPTIONAL_ACTIONS_ENABLED:
        raise PermissionError(f'unsupported optional receipt action: {action}')
    if OPTIONAL_ACTIONS_ENABLED.get(action) is not True:
        raise PermissionError(f'optional action is disabled by default: {action}')
    if os.environ.get('X1_OPERATOR_AUTHORIZATION', '') != OPTIONAL_AUTHORIZATION_PHRASE:
        raise PermissionError(f'X1_OPERATOR_AUTHORIZATION must equal {OPTIONAL_AUTHORIZATION_PHRASE}')
    normalized = [str(item) for item in arguments]
    options: dict[str, str] = {}
    index = 0
    while index < len(normalized):
        token = normalized[index]
        if not token.startswith('--'):
            raise ValueError('optional coordinator arguments must be explicit option/value pairs')
        if index + 1 >= len(normalized) or normalized[index + 1].startswith('--'):
            raise ValueError(f'missing value for option: {token}')
        if token in options:
            raise ValueError(f'duplicate option: {token}')
        options[token] = normalized[index + 1]
        index += 2
    missing_options = OPTIONAL_REQUIRED_OPTIONS[action] - set(options)
    if missing_options:
        raise ValueError(f'missing required options: {sorted(missing_options)}')
    for option in OPTIONAL_FILE_OPTIONS:
        if option in options and not Path(options[option]).expanduser().is_file():
            raise FileNotFoundError(f'{option} must name an existing file: {options[option]}')
    for option in OPTIONAL_DIRECTORY_OPTIONS:
        if option in options and not Path(options[option]).expanduser().is_dir():
            raise FileNotFoundError(f'{option} must name an existing directory: {options[option]}')
    output = Path(options['--out']).expanduser().resolve()
    try:
        output.relative_to(RUN_ROOT)
    except ValueError as exc:
        raise ValueError('--out must be inside the run root') from exc
    if action == 'commit-phase' and options.get('--phase') not in {'BASIS_QUALIFIED', 'PREDICT_COMMITTED', 'REVEAL_ACCEPTED'}:
        raise ValueError('commit-phase requires an allowed custody phase')
    name = f'optional_{action.replace("-", "_")}'
    invocation = run_coordinator_cli(
        name,
        normalized,
        accepted_returncodes=(0, 2),
        expected_statuses=('BLOCKED', 'ANALYSIS_BLOCKED', 'INCONCLUSIVE', 'QUALIFIED', 'NOT_QUALIFIED'),
    )
    OPTIONAL_ACTION_INVOCATIONS[action] = invocation
    return invocation

print(json.dumps({
    'optional_actions_enabled': OPTIONAL_ACTIONS_ENABLED,
    'invoked': False,
    'authorization_phrase': OPTIONAL_AUTHORIZATION_PHRASE,
    'model_execution': False,
    'training': False,
}, indent=2, sort_keys=True))


In [ ]:
def receipt_from(invocation: Any) -> dict[str, Any] | None:
    receipt = invocation.get('receipt') if isinstance(invocation, dict) else None
    return receipt if isinstance(receipt, dict) else None

protocol_receipt = receipt_from(globals().get('PROTOCOL_INVOCATION'))
inventory_receipt = receipt_from(globals().get('INVENTORY_INVOCATION'))
intake_receipt = receipt_from(globals().get('INTAKE_INVOCATION'))
gate_receipt = receipt_from(globals().get('GATE_INVOCATION'))
preflight_receipt = receipt_from(globals().get('PREFLIGHT_INVOCATION'))
plan_receipt = receipt_from(globals().get('EXECUTION_PLAN_INVOCATION'))
FINAL_BLOCKERS = list(ACTIVE_BLOCKERS)
if protocol_receipt is None:
    add_unique(FINAL_BLOCKERS, 'PROTOCOL_VALIDATION_NOT_RUN')
elif protocol_receipt.get('valid') is not True:
    add_unique(FINAL_BLOCKERS, 'PROTOCOL_INVALID')
if inventory_receipt is None:
    add_unique(FINAL_BLOCKERS, 'CHECKPOINT_INVENTORY_NOT_RUN')
elif inventory_receipt.get('eligible_candidates') == 0:
    add_unique(FINAL_BLOCKERS, 'NO_ELIGIBLE_CHECKPOINT')
if intake_receipt is None:
    add_unique(FINAL_BLOCKERS, 'CANDIDATE_INTAKE_NOT_RUN')
elif intake_receipt.get('status') != 'INVENTORIED':
    add_unique(FINAL_BLOCKERS, 'CANDIDATE_INTAKE_INCOMPLETE')
if gate_receipt is None:
    add_unique(FINAL_BLOCKERS, 'PREREQUISITE_GATE_NOT_RUN')
elif gate_receipt.get('status') != 'READY_FOR_EXTERNAL_PHASE_1':
    for blocker in gate_receipt.get('blockers', []):
        add_unique(FINAL_BLOCKERS, str(blocker))
if preflight_receipt is None:
    add_unique(FINAL_BLOCKERS, 'PREFLIGHT_NOT_RUN')
else:
    for blocker in preflight_receipt.get('blockers', []):
        add_unique(FINAL_BLOCKERS, str(blocker))
if plan_receipt is None:
    add_unique(FINAL_BLOCKERS, 'EXECUTION_PLAN_NOT_RUN')
else:
    for blocker in plan_receipt.get('blockers', []):
        add_unique(FINAL_BLOCKERS, str(blocker))
for invocation_name in (
    'PROTOCOL_INVOCATION', 'INVENTORY_INVOCATION', 'INTAKE_INVOCATION', 'GATE_INVOCATION',
    'PREFLIGHT_INVOCATION', 'EXECUTION_PLAN_INVOCATION',
):
    invocation = globals().get(invocation_name)
    if isinstance(invocation, dict) and not invocation.get('unavailable') and invocation.get('accepted_return_code') is False:
        add_unique(FINAL_BLOCKERS, f'UNEXPECTED_CLI_RESULT:{invocation_name}')
add_unique(FINAL_BLOCKERS, 'SEPARATE_RELEASED_EXTERNAL_RUNNER_REQUIRED')
add_unique(FINAL_BLOCKERS, 'TFG_UNRESOLVED')
if RUNTIME_REPORT.get('accelerator', {}).get('kind') == 'CPU':
    add_unique(FINAL_BLOCKERS, 'CURRENT_COLAB_RUNTIME_IS_CPU')

def regular_file_inventory(root: Path | None) -> dict[str, Any]:
    if root is None or not root.is_dir():
        return {'root': str(root) if root else None, 'files': 0, 'bytes': 0, 'symlinks': 0}
    files = 0
    size = 0
    symlinks = 0
    for current, directories, filenames in os.walk(root, followlinks=False):
        current_path = Path(current)
        for directory in directories:
            if (current_path / directory).is_symlink():
                symlinks += 1
        for filename in filenames:
            path = current_path / filename
            if path.is_symlink():
                symlinks += 1
            elif path.is_file():
                files += 1
                size += path.stat().st_size
    return {'root': str(root), 'files': files, 'bytes': size, 'symlinks': symlinks}

FINAL_STATUS_REPORT = {
    'schema': 'x1-real-1-colab-final-status/v1',
    'status': 'BLOCKED' if FINAL_BLOCKERS else 'MODEL_FREE_PREPARATION_REPORTED',
    'blockers': FINAL_BLOCKERS,
    'guards': [
        'CANDIDATE_INTAKE_NEVER_PROMOTES',
        'NO_MODEL_DESERIALIZATION',
        'NO_MODEL_INFERENCE',
        'NO_TRAINING',
        'NO_AUTOMATIC_DOWNLOAD',
        'NO_AUTOMATIC_COHORT_OR_PREDICTOR_FIT',
        'NO_AUTOMATIC_OUTCOME_OR_VERIFIER_EXECUTION',
    ],
    'protocol': {
        'path': str(PROTOCOL_PATH) if PROTOCOL_PATH else None,
        'declared_sha256': PROTOCOL_DOC.get('identity', {}).get('protocol_sha256') if isinstance(PROTOCOL_DOC, dict) else None,
        'validated_sha256': protocol_receipt.get('protocol_sha256') if protocol_receipt else None,
        'source_closure_checked': True,
        'validation': protocol_receipt,
    },
    'runtime': RUNTIME_REPORT,
    'tfg': {'status': 'UNRESOLVED', 'meaning_assumed': False},
    'storage': {
        'content_disk': RUNTIME_REPORT.get('storage'),
        'artifact_root': regular_file_inventory(ARTIFACT_ROOT),
        'run_root': regular_file_inventory(RUN_ROOT),
        'checkpoint': {
            'path': str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None,
            'bytes': CHECKPOINT_PATH.stat().st_size if CHECKPOINT_PATH else None,
            'sha256': intake_receipt.get('checkpoint_file_sha256') if intake_receipt else None,
            'packaged': False,
        },
    },
    'candidate_intake': intake_receipt,
    'inventory': inventory_receipt,
    'gate': gate_receipt,
    'preflight': preflight_receipt,
    'execution_plan': plan_receipt,
    'optional_receipt_only_invocations': OPTIONAL_ACTION_INVOCATIONS,
    'scientific_campaign': {
        'status': 'NOT_RUN',
        'cohort_generator': False,
        'predictor_fitter': False,
        'outcome_runner': False,
        'verifier_runner': False,
        'analysis': False,
    },
    'external_released_runner_required': True,
    'external_runner_present_in_notebook': False,
    'local_model_execution': False,
    'training': False,
}
FINAL_STATUS_RECEIPT, FINAL_STATUS_RESUMED = persist_receipt('final_status.json', FINAL_STATUS_REPORT)
print(json.dumps({'final_status': FINAL_STATUS_RECEIPT, 'resumed': FINAL_STATUS_RESUMED}, indent=2, sort_keys=True))


In [ ]:
import zipfile

def package_failure(reason: str, details: Any = None) -> dict[str, Any]:
    report = {
        'schema': 'x1-real-1-colab-package-failure/v1',
        'status': 'BLOCKED',
        'reason': reason,
        'details': details,
        'local_model_execution': False,
        'training': False,
    }
    persisted, _ = persist_receipt('package_failure.json', report)
    print(json.dumps(persisted, indent=2, sort_keys=True))
    return persisted

def archive_token() -> str:
    if SAFE_TOKEN.fullmatch(RUN_ROOT.name):
        return RUN_ROOT.name
    return hashlib.sha256(str(RUN_ROOT).encode('utf-8')).hexdigest()[:24]

def verify_archive(path: Path, expected_members: dict[str, dict[str, Any]]) -> tuple[bool, str, dict[str, dict[str, Any]]]:
    observed: dict[str, dict[str, Any]] = {}
    with zipfile.ZipFile(path, 'r') as bundle:
        bad_member = bundle.testzip()
        if bad_member is not None:
            return False, f'ZIP CRC failure at {bad_member}', observed
        names = bundle.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected_members):
            return False, 'ZIP member set is not exact', observed
        for name in sorted(names):
            if name.startswith('/') or '..' in Path(name).parts:
                return False, f'unsafe ZIP member: {name}', observed
            payload = bundle.read(name)
            digest = hashlib.sha256(payload).hexdigest()
            observed[name] = {'bytes': len(payload), 'sha256': digest}
            if observed[name] != expected_members[name]:
                return False, f'ZIP member hash or size mismatch: {name}', observed
    return True, None, observed

PACKAGE_RESULT: dict[str, Any]
try:
    package_files: list[dict[str, Any]] = []
    package_paths: list[Path] = []
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_symlink():
            raise ReceiptConflictError(f'run root contains a symlink: {path}')
        if path.is_file():
            relative = path.relative_to(RUN_ROOT).as_posix()
            if relative != 'package_inventory.json':
                package_files.append({'path': relative, 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
            package_paths.append(path)
    inventory_document = {
        'schema': 'x1-real-1-colab-package-inventory/v1',
        'run_root': str(RUN_ROOT),
        'files': package_files,
        'checkpoint_weights_included': False,
    }
    persisted_inventory, inventory_resumed = persist_receipt('package_inventory.json', inventory_document)
    if persisted_inventory.get('files') != package_files or persisted_inventory.get('run_root') != str(RUN_ROOT):
        raise ReceiptConflictError('existing package_inventory.json does not match the current immutable run root')
    expected_members: dict[str, dict[str, Any]] = {}
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_symlink():
            raise ReceiptConflictError(f'run root contains a symlink: {path}')
        if path.is_file():
            member = f'{RUN_ROOT.name}/{path.relative_to(RUN_ROOT).as_posix()}'
            expected_members[member] = {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
    token = archive_token()
    archive_path = CONTENT_ROOT / f'x1-real-1-{token}-receipts.zip'
    partial_path = CONTENT_ROOT / f'.x1-real-1-{token}-receipts.partial.zip'
    receipt_path = CONTENT_ROOT / f'x1-real-1-{token}-receipts.zip.sha256.json'
    if not archive_path.exists():
        if partial_path.exists():
            raise ReceiptConflictError(f'partial archive already exists; it will not be overwritten or deleted: {partial_path}')
        with zipfile.ZipFile(partial_path, 'x', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
            for member in sorted(expected_members):
                source_relative = member.split('/', 1)[1]
                bundle.write(RUN_ROOT / source_relative, arcname=member)
        verified, verification_error, observed_members = verify_archive(partial_path, expected_members)
        if not verified:
            raise ReceiptConflictError(verification_error)
        os.link(partial_path, archive_path)
        partial_path.unlink()
    verified, verification_error, observed_members = verify_archive(archive_path, expected_members)
    if not verified:
        raise ReceiptConflictError(verification_error)
    digest = sha256_file(archive_path)
    archive_receipt = {
        'schema': 'x1-real-1-colab-receipt-archive/v1',
        'status': 'VERIFIED',
        'archive': archive_path.name,
        'archive_path': str(archive_path),
        'sha256': digest,
        'bytes': archive_path.stat().st_size,
        'run_root': str(RUN_ROOT),
        'run_id': RUN_ID,
        'file_count': len(observed_members),
        'members': observed_members,
        'zip_integrity_verified': True,
        'checkpoint_weights_included': False,
        'local_model_execution': False,
        'training': False,
    }
    if receipt_path.exists():
        existing_archive_receipt = strict_json_load(receipt_path)
        if existing_archive_receipt != archive_receipt:
            raise ReceiptConflictError('existing archive SHA-256 receipt does not match the verified archive')
        archive_receipt = existing_archive_receipt
        receipt_resumed = True
    else:
        write_json_no_overwrite(receipt_path, archive_receipt)
        receipt_resumed = False
    PACKAGE_RESULT = {
        'status': 'VERIFIED',
        'archive': str(archive_path),
        'sha256': digest,
        'receipt': str(receipt_path),
        'file_count': len(observed_members),
        'package_inventory_resumed': inventory_resumed,
        'archive_receipt_resumed': receipt_resumed,
        'checkpoint_weights_included': False,
        'local_model_execution': False,
        'training': False,
    }
except Exception as exc:
    PACKAGE_RESULT = package_failure(f'{type(exc).__name__}: {exc}', {'run_root': str(RUN_ROOT)})
print(json.dumps({'package': PACKAGE_RESULT}, indent=2, sort_keys=True))
if PACKAGE_RESULT.get('status') == 'VERIFIED':
    try:
        from IPython.display import FileLink, display
        display(FileLink(PACKAGE_RESULT['archive']))
    except ImportError:
        print('IPython FileLink is unavailable; download the archive from the Colab file pane.')


## Receipt download and interpretation

The final code cell creates the archive through a uniquely named partial file, verifies every member and the ZIP CRC, computes the archive SHA-256, promotes the partial file without overwriting an existing archive, and writes a no-overwrite SHA-256 receipt beside it. It packages only files under the run root; checkpoint weights and supplied source, registry, and artifact files are not copied into the archive. `FileLink` is used when IPython exposes it; otherwise use the Colab file pane.

Treat `BLOCKED` as the expected fail-closed result when artifacts, eligibility, custody, or source identity are incomplete. A generated execution plan still does not authorize external compute. Do not claim that X1-REAL-1 ran scientifically unless a separate released external runner and its own receipt chain independently establish that execution.